In [0]:
from pyspark.sql import functions as F

### master join function

In [0]:
products_path = "/Volumes/workspace/default/olist_files/olist_products_dataset.csv"
silver_products_df = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load(products_path) \
    .select("product_id", "product_category_name", "product_weight_g") \
    .filter(F.col("product_category_name").isNotNull())

In [0]:
display(silver_products_df.limit(10))

product_id,product_category_name,product_weight_g
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,225
3aa071139cb16b67ca9e5dea641aaa2f,artes,1000
96bd76ec8810374ed1b65e291975717f,esporte_lazer,154
cef67bcfe19066a932b7673e239eb23d,bebes,371
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,625
41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,200
732bd381ad09e530fe0a5f457d81becb,cool_stuff,18350
2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao,900
37cc742be07708b53a98702e77a21a02,eletrodomesticos,400
8c92109888e8cdf9d66dc7e463025574,brinquedos,600


In [0]:
translation_path = "/Volumes/workspace/default/olist_files/product_category_name_translation.csv"

translation_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(translation_path)

In [0]:
display(translation_df.limit(10))

product_category_name,product_category_name_english
beleza_saude,health_beauty
informatica_acessorios,computers_accessories
automotivo,auto
cama_mesa_banho,bed_bath_table
moveis_decoracao,furniture_decor
esporte_lazer,sports_leisure
perfumaria,perfumery
utilidades_domesticas,housewares
telefonia,telephony
relogios_presentes,watches_gifts


### learn left join in pyspark

In [0]:

joined_df = silver_products_df.join(
    translation_df, 
    on="product_category_name", 
    how="left"
)

In [0]:
final_products_df = joined_df.withColumn(
    "product_category_name_english",
    F.coalesce(
        F.col("product_category_name_english"),
        F.lit("Unknown")
        )
)

print(f"名称空值的个数为{final_products_df \
    .filter(F.col('product_category_name_english').isNull()) \
    .count()}")

名称空值的个数为0


### learn how to use groupBy() function and .sort() function

In [0]:
# calculate the number of products in each category
report_df = final_products_df \
    .groupBy("product_category_name_english") \
    .agg(F.count("product_id").alias("total_products")) \
    .sort(F.col("total_products").desc())

# display the top 10
display(report_df.limit(10))


product_category_name_english,total_products
bed_bath_table,3029
sports_leisure,2867
furniture_decor,2657
health_beauty,2444
housewares,2335
auto,1900
computers_accessories,1639
toys,1411
watches_gifts,1329
telephony,1134


### study how to count duplicate values

step1: dataset.count()

step2: dataset.select("field name").distinct().count()



In [0]:
orders_path = "/Volumes/workspace/default/olist_files/olist_order_items_dataset.csv"

order_items_df = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load(orders_path)


total_count = order_items_df.count()
unique_order_ids = order_items_df.select("order_id").distinct().count()

print(f"There are {total_count} rows in the order_items dataset and {unique_order_ids} are unique order IDs")

There are 112650 rows in the order_items dataset and 98666 are unique order IDs


### study how to remove duplicate values

.dropDuplicates(['filed name'])

In [0]:
order_items_clean_df = order_items_df \
                            .filter(F.col("order_item_id").isNotNull()) \
                            .dropDuplicates(["order_id"])

total_count_clean = order_items_clean_df.count()
unique_order_ids_clean = order_items_clean_df.select("order_id").distinct().count()

print(f"There are {total_count_clean} rows in the order_items dataset after cleaning and {unique_order_ids_clean} are unique order IDs")

There are 98666 rows in the order_items dataset after cleaning and 98666 are unique order IDs
